# Equivalenze

In [ ]:
from QuantumCompiling import CircuitState, CircuitShape
import scipy.sparse as sp
import numpy as np
import itertools as it

In [ ]:
n_qubits = 4
gates = { 
    1 : [ ["ID"] ],
    2 : [ ["Sc","St"],["CZc","CZt"] ],
    3 : [ ["Sc","ID","St"] ],
    4 : [ ["Sc","Sc","St","St"] ] ,
}

shape = CircuitShape(n_qubits,gates)
shape.print_steps(); 

d = 1, nessun time-step è equivalente a sè stesso

In [ ]:
available_steps = shape.get_steps()
for i in range(len(available_steps)):
    c1 = CircuitState(shape,[available_steps[i]])
    for j in range(i+1,len(available_steps)):
        c2 = CircuitState(shape,[available_steps[j]])
        if(CircuitState.equivalent(c1,c2)):
            print(c1.draw())
            print(c2.draw())

d=2, si costruiscono tutti i circuiti a due step (effettivi, la classe CircuitState associa lo stesso stato a quelli banalmente equivalenti e se possibile semplifica ad un numero minore di timesteps; va dunque trattato un solo circuito per classe di banale equivalenza e quelli contraibili vanno scartati) e si confrontano tra loro

In [ ]:
available_steps = shape.get_steps()

circuits = []
ids = []

for i in range(len(available_steps)):
    for j in range(len(available_steps)):
        c = CircuitState(shape, [available_steps[i],available_steps[j]])
        if c.get_n_steps() == 2 and c not in ids: 
            if not c.isID():
                if all(not c.equals(c1) for c1 in circuits):
                    circuits.append(c)
            else:
                ids.append(c)
                
print("Circuiti effettivi:", len(circuits))
# for c in circuits: print(c.draw())
# print()
# print()
print("Identità:", len(ids))
# for c in ids: print(c.draw())

In [ ]:
n_eq = 0
n_confronti = 0
for i in range(len(circuits)-1):
    for j in range(i+1, len(circuits)):
        n_confronti += 1
        c1 = circuits[i]
        c2 = circuits[j]
        if CircuitState.equivalent(c1,c2): n_eq +=1
        # print("Contfronto tra:",CircuitState.equivalent(c1,c2))
        # print(c1.draw())
        # print(c2.draw())
        # print()
print("Numero di confronti:",n_confronti)
print("Numero di equivalenze:",n_eq)

In generale per d timesteps

In [ ]:
d=2

actual_circuits = []
IDequivalent = []
available_steps = shape.get_steps()

ids = list(it.product(range(len(available_steps)), repeat=d))
for i in range(len(ids)):
    c = CircuitState(shape, [available_steps[k] for k in ids[i]])
    if c.get_n_steps() == d and c not in IDequivalent:
        if c.isID():
            IDequivalent.append(c)
        elif all(not c.equals(c1) for c1 in actual_circuits):
            actual_circuits.append(c)
print("Circuiti effettivi trovati:",len(actual_circuits))
print("Circuiti identità:",len(IDequivalent))

act = [10,66,568,4050]
id = [1,10,0,350]

Per trovare le equivalenze ora basta semplicemente confrontare ogni circuito, escludendo l'ultimo, con tutti i successivi della lista

In [ ]:
# n_eq = 0
# for i in range(len(actual_circuits)-1):
#     for j in range(i+1,len(actual_circuits)):
#         if CircuitState.equivalent(actual_circuits[i],actual_circuits[j]):
#             n_eq += 1
# print("Numero di equivalenze:",n_eq)

Le equivalenze vanno salvate su file, per farlo si usano gli indici i e j. In questo modo il file dipende strettamente dalla shape utilizzata, per questo anch'essa deve stare nel file

In [ ]:
import csv

filename = "csv/eq2_c.csv"
with open(filename, 'w', newline='') as fout:
    writer = csv.writer(fout, delimiter="\t")
    writer.writerow(["n_qubits:" +str(n_qubits)])
    writer.writerow(["gates:"])
    for key in gates.keys():
        writer.writerow(gates[key])
    writer.writerow(["equivalences:"])
    for i in range(len(actual_circuits)-1):
        print("C1:",str(i+1)+"/"+str(len(actual_circuits)-1))
        for j in range(i+1,len(actual_circuits)):
            if CircuitState.equivalent(actual_circuits[i],actual_circuits[j]):
                writer.writerow( [shape.get_step_index(step) for step in actual_circuits[i].get_steps()]
                                + [shape.get_step_index(step) for step in actual_circuits[j].get_steps()])

Si sposta il codice nel metodo find_equivalences della classe CircuitShape

In [ ]:
n_qubits = 4
gates = { 
    1 : [ ["ID"] ],
    2 : [ ["Sc","St"],["CZc","CZt"] ]
    # ,3 : [ ["Sc","ID","St"] ],
    # 4 : [ ["Sc","Sc","St","St"] ] ,
}

CircuitShape.find_equivalences(gates,n_qubits,6,"csv_lungo")